# Servicios cercanos con Google Places API

Para cada aviso con coordenadas, averigua qué servicios tiene alrededor —metro, colegios,
supermercados, áreas verdes, salud— y guarda el resultado en la base en una columna nueva llamada
`servicios_cercanos`.

El objetivo es alimentar los modelos de precio: **la ubicación es, junto con la superficie, el
determinante principal del valor de una vivienda**, y hoy los modelos solo la conocen por las
coordenadas crudas y la comuna.

## ¿Alcanza la cuota gratuita? Sí, con margen

### Cómo cobra Google Maps Platform desde marzo de 2025

El antiguo crédito de 200 USD mensuales desapareció. Ahora **cada SKU tiene su propio tope gratuito
mensual**:

| Nivel | Llamadas gratis por SKU al mes |
|---|---|
| Essentials | 10.000 |
| **Pro** — aquí cae *Nearby Search* | **5.000** |
| Enterprise | 1.000 |

Aparte, una cuenta nueva de Google Cloud recibe **300 USD de crédito por 90 días**.

### Cuántas llamadas necesitamos

El dataset tiene 15.808 avisos, pero solo **14.348 traen coordenadas** (los 1.460 de
ChilePropiedades no tienen). Y muchos comparten ubicación por estar en el mismo edificio:

| Resolución | Puntos únicos | Ahorro |
|---|---|---|
| Coordenada exacta | 8.052 | 44 % |
| 4 decimales (~11 m) | 7.514 | 48 % |
| **3 decimales (~110 m)** | **3.453** | **76 %** |

Usamos la grilla de **3 decimales**: dos propiedades separadas por 110 metros tienen prácticamente
los mismos servicios alrededor, así que consultar ambas sería pagar dos veces la misma respuesta.

### La estrategia de dos niveles, y por qué

La primera versión de este notebook pedía todos los tipos de servicio en una sola llamada por
punto. **Al probarlo con datos reales, no funcionó.** La API devuelve como máximo 20 resultados, y
en una consulta con todos los tipos a la vez el resultado fue este:

```
20 resultados -> 9 consultorios, 3 farmacias, 2 malls, 1 banco...
                 ningún metro, ningún colegio, ningún parque
```

Las categorías **densas** (farmacias, consultorios) inundan el cupo y tapan a las **escasas**
(metro), que son justamente las que más pesan en el precio. Consultando solo transporte, en cambio,
aparecen 3 estaciones sin problema.

De ahí el diseño en dos niveles:

| | Categorías | Método | Llamadas |
|---|---|---|---|
| **Nivel 1** | Metro, hospitales, malls, universidades | Se enumeran **una sola vez** sobre toda el área con una grilla, y luego se calcula la distancia en local | **~64** |
| **Nivel 2** | Colegios, parques, supermercados | Una llamada **por punto**: son demasiado densos para enumerarlos | **3.453** |

**Total ≈ 3.520 llamadas, bajo las 5.000 gratuitas. Costo esperado: cero.**

El nivel 1 no solo es más barato: es **más exacto**. Con 16 llamadas se obtienen las 84 estaciones
de metro del área, y a partir de ahí todos los avisos tienen su distancia real al metro más
cercano. Con el método por punto, un aviso solo habría sabido del metro si este lograba colarse
entre los 20 resultados más cercanos, cosa que casi nunca pasa.

In [ ]:
import json
import os
import shutil
import sqlite3
import time
from datetime import datetime
from pathlib import Path

import httpx
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# override=True es OBLIGATORIO: sin el, load_dotenv NO reemplaza una variable que
# ya este en memoria del kernel. Si editas el .env y vuelves a ejecutar esta celda
# sin override, seguirias usando la clave anterior sin darte cuenta.
# Ruta explicita ademas, porque load_dotenv() sin argumentos falla en algunos contextos.
load_dotenv(".env", override=True)
API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "").strip()

DB = Path("data/ofertas_unificado.sqlite")
CACHE = Path("data/poi_cache.sqlite")      # respuestas crudas, para no repetir llamadas
ENDPOINT = "https://places.googleapis.com/v1/places:searchNearby"

RADIO_M = 1200          # 1,2 km: distancia caminable razonable en Santiago
MAX_RESULTADOS = 20     # tope duro de la API
DECIMALES = 3           # grilla de ~110 m

if not API_KEY:
    print("FALTA la clave. Agrega esta linea al archivo .env de la raiz del proyecto:")
    print("   GOOGLE_MAPS_API_KEY=tu_clave_aqui")
else:
    print(f"clave en uso: {API_KEY[:8]}...{API_KEY[-4:]}  ({len(API_KEY)} caracteres)")
    print("Comprueba que estos caracteres coincidan con la clave que quieres usar.")

In [2]:
# --- NIVEL 1: escasas y de ubicacion fija -> se enumeran una vez ---------------
CATALOGO = {
    "metro":       ["subway_station", "light_rail_station"],
    "hospital":    ["hospital"],
    "mall":        ["shopping_mall"],
    "universidad": ["university"],
}

# --- NIVEL 2: densas -> una llamada por punto ---------------------------------
# Se excluyen farmacias, bancos y consultorios a proposito: son tan densos que
# llenarian los 20 resultados, y su efecto sobre el precio es marginal.
DENSAS = ["school", "primary_school", "secondary_school", "park", "supermarket"]

GRUPOS_DENSOS = {
    "colegio":      {"school", "primary_school", "secondary_school", "preschool"},
    "parque":       {"park", "city_park", "dog_park", "state_park", "national_park"},
    "supermercado": {"supermarket", "grocery_store"},
}

# Caja que contiene las 4 comunas, para la grilla del nivel 1
GRILLA_LAT = np.linspace(-33.61, -33.41, 4)
GRILLA_LON = np.linspace(-70.71, -70.49, 4)
RADIO_GRILLA_M = 3500

print(f"NIVEL 1 (catalogo): {list(CATALOGO)}")
print(f"   {len(GRILLA_LAT)*len(GRILLA_LON)} puntos de grilla x {len(CATALOGO)} categorias "
      f"= {len(GRILLA_LAT)*len(GRILLA_LON)*len(CATALOGO)} llamadas")
print(f"\nNIVEL 2 (por punto): {list(GRUPOS_DENSOS)}")
print(f"   ~3.453 llamadas, una por punto unico")

NIVEL 1 (catalogo): ['metro', 'hospital', 'mall', 'universidad']
   16 puntos de grilla x 4 categorias = 64 llamadas

NIVEL 2 (por punto): ['colegio', 'parque', 'supermercado']
   ~3.453 llamadas, una por punto unico


## Funciones base

Toda respuesta cruda se guarda en `data/poi_cache.sqlite` **antes** de procesarla. Si el proceso se
corta, al relanzarlo solo consulta lo que falta: nunca se gasta cuota dos veces por lo mismo.

In [3]:
CAMPOS = "places.displayName,places.types,places.primaryType,places.location"


def buscar(lat, lon, tipos, radio, cliente):
    """Una llamada a Nearby Search, ordenada por distancia."""
    cuerpo = {
        "includedTypes": list(tipos),
        "maxResultCount": MAX_RESULTADOS,
        "rankPreference": "DISTANCE",
        "locationRestriction": {"circle": {
            "center": {"latitude": float(lat), "longitude": float(lon)},
            "radius": float(radio)}},
        "languageCode": "es",
    }
    cabeceras = {"Content-Type": "application/json", "X-Goog-Api-Key": API_KEY,
                 # El fieldMask decide que se cobra: pedir solo lo necesario
                 "X-Goog-FieldMask": CAMPOS}
    r = cliente.post(ENDPOINT, json=cuerpo, headers=cabeceras, timeout=30)
    r.raise_for_status()
    return r.json().get("places", [])


def metros(lat1, lon1, lat2, lon2):
    """Distancia haversine en metros. Vectorizada: acepta arrays."""
    R = 6_371_000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def iniciar_cache():
    with sqlite3.connect(CACHE) as con:
        con.execute("""CREATE TABLE IF NOT EXISTS puntos (
            punto TEXT PRIMARY KEY, lat REAL, lon REAL,
            respuesta TEXT, estado TEXT, cuando TEXT)""")
        con.execute("""CREATE TABLE IF NOT EXISTS catalogo (
            categoria TEXT, nombre TEXT, lat REAL, lon REAL,
            PRIMARY KEY (categoria, lat, lon))""")


iniciar_cache()
print(f"cache: {CACHE}")

cache: data\poi_cache.sqlite


## Nivel 1 — Catálogo global de servicios escasos

Se barre el área con una grilla de 4×4 puntos y radio de 3,5 km, una vez por categoría. Son 64
llamadas en total y quedan enumerados todos los metros, hospitales, malls y universidades de las
cuatro comunas y sus alrededores.

Después la distancia de cada aviso a cada categoría se calcula **en local**, sin costo y de forma
exacta.

In [4]:
def construir_catalogo(forzar=False):
    with sqlite3.connect(CACHE) as con:
        ya = {r[0] for r in con.execute("SELECT DISTINCT categoria FROM catalogo")}

    pendientes = [c for c in CATALOGO if forzar or c not in ya]
    if not pendientes:
        print("catalogo ya construido (usa forzar=True para rehacerlo)")
        return

    llamadas = 0
    with httpx.Client() as cliente:
        for categoria in pendientes:
            encontrados = {}
            for la in GRILLA_LAT:
                for lo in GRILLA_LON:
                    try:
                        for p in buscar(la, lo, CATALOGO[categoria], RADIO_GRILLA_M, cliente):
                            loc = p["location"]
                            clave = (round(loc["latitude"], 5), round(loc["longitude"], 5))
                            encontrados[clave] = p.get("displayName", {}).get("text", "")
                    except httpx.HTTPStatusError as e:
                        print(f"   error {e.response.status_code} en {categoria}")
                    llamadas += 1
                    time.sleep(0.05)
            with sqlite3.connect(CACHE) as con:
                con.executemany("INSERT OR REPLACE INTO catalogo VALUES (?,?,?,?)",
                                [(categoria, n, la, lo) for (la, lo), n in encontrados.items()])
            print(f"   {categoria:<12} {len(encontrados):>4} lugares")
    print(f"\n{llamadas} llamadas usadas")


if API_KEY:
    construir_catalogo()

with sqlite3.connect(CACHE) as con:
    catalogo = pd.read_sql_query("SELECT * FROM catalogo", con)
print(f"\ncatalogo: {len(catalogo):,} lugares")
print(catalogo["categoria"].value_counts().to_string())

   metro          84 lugares


   hospital      226 lugares


   mall          244 lugares


   universidad   190 lugares

64 llamadas usadas

catalogo: 744 lugares
categoria
mall           244
hospital       226
universidad    190
metro           84


## Nivel 2 — Los puntos únicos a consultar

Se agrupan los avisos por coordenada redondeada a 3 decimales. Cada punto se consulta una vez y su
respuesta se reparte después a todos los avisos que caen en esa celda.

In [5]:
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True, timeout=15) as con:
    avisos = pd.read_sql_query("SELECT sitio, id_aviso, comuna, lat, lon FROM ofertas", con)

avisos = avisos.replace("SIN_DATO", pd.NA)
for c in ("lat", "lon"):
    avisos[c] = pd.to_numeric(avisos[c], errors="coerce")

con_geo = avisos[avisos["lat"].notna() & avisos["lon"].notna()].copy()
con_geo["punto"] = (con_geo["lat"].map(f"{{:.{DECIMALES}f}}".format) + ","
                    + con_geo["lon"].map(f"{{:.{DECIMALES}f}}".format))

puntos = (con_geo.groupby("punto")
          .agg(lat=("lat", "mean"), lon=("lon", "mean"), avisos=("id_aviso", "size"))
          .reset_index())

with sqlite3.connect(CACHE) as con:
    hechos = {r[0] for r in con.execute("SELECT punto FROM puntos WHERE estado='ok'")}

print(f"avisos totales         : {len(avisos):,}")
print(f"con coordenadas        : {len(con_geo):,}")
print(f"sin coordenadas        : {len(avisos)-len(con_geo):,}  (quedaran en SIN_DATO)")
print(f"\nPUNTOS UNICOS          : {len(puntos):,}")
print(f"ya consultados         : {len(hechos):,}")
print(f"PENDIENTES             : {len(puntos)-len(hechos):,}")
print(f"\ncada punto cubre en promedio {len(con_geo)/len(puntos):.1f} avisos")

avisos totales         : 15,808
con coordenadas        : 14,348
sin coordenadas        : 1,460  (quedaran en SIN_DATO)

PUNTOS UNICOS          : 3,453
ya consultados         : 0
PENDIENTES             : 3,453

cada punto cubre en promedio 4.2 avisos


## Prueba con 2 puntos

Antes de lanzar 3.453 llamadas, confirmar que la clave responde y que las tres categorías densas se
reparten los 20 resultados de forma razonable.

In [6]:
def clasificar(lugar):
    """Devuelve las categorias densas a las que pertenece un lugar."""
    tipos = set(lugar.get("types", []))
    return {g for g, ts in GRUPOS_DENSOS.items() if tipos & ts}


if not API_KEY:
    print("Sin clave en .env: no se puede probar.")
else:
    with httpx.Client() as cliente:
        for _, p in puntos.head(2).iterrows():
            print(f"\n=== {p['punto']}  ({int(p['avisos'])} avisos) ===")
            try:
                lugares = buscar(p["lat"], p["lon"], DENSAS, RADIO_M, cliente)
                reparto = {}
                for l in lugares:
                    for g in clasificar(l):
                        reparto[g] = reparto.get(g, 0) + 1
                print(f"{len(lugares)} resultados -> {reparto}")
                for l in lugares[:6]:
                    d = metros(p["lat"], p["lon"],
                               l["location"]["latitude"], l["location"]["longitude"])
                    cats = ",".join(clasificar(l)) or "-"
                    print(f"   {d:>6.0f} m  [{cats:<12}] "
                          f"{l.get('displayName',{}).get('text','?')[:45]}")
            except httpx.HTTPStatusError as e:
                print(f"ERROR {e.response.status_code}: {e.response.text[:400]}")
                break


=== -33.435,-70.581  (3 avisos) ===


20 resultados -> {'colegio': 12, 'supermercado': 4, 'parque': 4}
      102 m  [colegio     ] Casa Nativo
      126 m  [supermercado] OXXO
      252 m  [colegio     ] Casa Aprendizaje Little House
      270 m  [colegio     ] Jardín Infantil Qantati
      288 m  [colegio     ] Jardin Infantil y Sala Cuna Amancay
      288 m  [parque      ] Parque Canal San Carlos

=== -33.435,-70.582  (2 avisos) ===


20 resultados -> {'supermercado': 4, 'colegio': 12, 'parque': 4}
       78 m  [supermercado] OXXO
       83 m  [colegio     ] Casa Nativo
      209 m  [colegio     ] Casa Aprendizaje Little House
      259 m  [colegio     ] Jardin Infantil y Sala Cuna Amancay
      295 m  [supermercado] Minimarket Comercial Ebenezer
      301 m  [colegio     ] Jardín Infantil Qantati


## Corrida completa del nivel 2

Reanudable: si se corta, vuelve a ejecutar esta celda y sigue donde quedó.

**`LIMITE`** acota cuántos puntos nuevos se consultan en esta pasada. Empieza con un número chico
para medir el ritmo; ponlo en `None` para procesar todo lo que falte.

In [ ]:
LIMITE = None        # None = todos los pendientes · un numero = solo esa cantidad
PAUSA_S = 0.05

with sqlite3.connect(CACHE) as con:
    hechos = {r[0] for r in con.execute("SELECT punto FROM puntos WHERE estado='ok'")}
pendientes = puntos[~puntos["punto"].isin(hechos)]
if LIMITE:
    pendientes = pendientes.head(LIMITE)

print(f"se consultaran {len(pendientes):,} puntos\n")
ok = fallos = 0
inicio = time.perf_counter()

if API_KEY and len(pendientes):
    with httpx.Client() as cliente, sqlite3.connect(CACHE) as con:
        for i, (_, p) in enumerate(pendientes.iterrows(), 1):
            try:
                lugares = buscar(p["lat"], p["lon"], DENSAS, RADIO_M, cliente)
                estado, carga = "ok", lugares
                ok += 1
            except httpx.HTTPStatusError as e:
                estado, carga = f"http_{e.response.status_code}", {"error": e.response.text[:400]}
                fallos += 1
            except Exception as e:
                estado, carga = "error", {"error": str(e)[:400]}
                fallos += 1

            con.execute("INSERT OR REPLACE INTO puntos VALUES (?,?,?,?,?,?)",
                        (p["punto"], p["lat"], p["lon"],
                         json.dumps(carga, ensure_ascii=False), estado,
                         datetime.now().isoformat(timespec="seconds")))
            con.commit()

            if estado.startswith("http_") and estado.split("_")[1] in ("401", "403", "429"):
                print(f"DETENIDO en {estado}: revisa la clave o la cuota")
                break
            if i % 100 == 0:
                seg = time.perf_counter() - inicio
                print(f"  {i:>5,}/{len(pendientes):,}   ok {ok:,}  fallos {fallos}   "
                      f"{i/seg:.1f} llamadas/s   quedan ~{(len(pendientes)-i)/(i/seg)/60:.1f} min")
            time.sleep(PAUSA_S)

with sqlite3.connect(CACHE) as con:
    total = con.execute("SELECT COUNT(*) FROM puntos WHERE estado='ok'").fetchone()[0]
print(f"\nok {ok:,} · fallos {fallos} · {time.perf_counter()-inicio:.1f}s")
print(f"cache: {total:,} de {len(puntos):,} puntos ({100*total/len(puntos):.1f}%)")

## Control de cuota

Esta celda cuenta las llamadas que **este notebook** hizo realmente, leyendo la caché. Sirve para
saber en qué punto del cupo mensual vas sin salir de Jupyter.

Es una cuenta local, no la oficial de Google: si consultas la misma API desde otro proyecto o
script, esos consumos no aparecen aquí. La cifra que manda es la de la consola de Google Cloud,
explicada en la celda de más abajo.

In [ ]:
GRATIS_MES = 5_000       # tope gratuito del SKU Pro (Nearby Search)
USD_POR_MIL = 32.00      # precio del tramo 0-100.000

with sqlite3.connect(CACHE) as con:
    por_dia = pd.read_sql_query(
        "SELECT substr(cuando,1,7) AS mes, substr(cuando,1,10) AS dia, "
        "COUNT(*) AS llamadas FROM puntos GROUP BY dia ORDER BY dia", con)
    n_catalogo_llamadas = len(GRILLA_LAT) * len(GRILLA_LON) * con.execute(
        "SELECT COUNT(DISTINCT categoria) FROM catalogo").fetchone()[0]

print(f"Nivel 1 (catalogo)  : {n_catalogo_llamadas:>6,} llamadas")
print(f"Nivel 2 (por punto) : {int(por_dia['llamadas'].sum()):>6,} llamadas")

if len(por_dia):
    print("\nPor dia:")
    print(por_dia[["dia", "llamadas"]].to_string(index=False))

    mes_actual = datetime.now().strftime("%Y-%m")
    del_mes = int(por_dia.loc[por_dia["mes"] == mes_actual, "llamadas"].sum())
    total_mes = del_mes + n_catalogo_llamadas
    cobrable = max(0, total_mes - GRATIS_MES)

    print(f"\n--- Mes en curso ({mes_actual}) ---")
    print(f"  llamadas    : {total_mes:>6,} de {GRATIS_MES:,} gratuitas "
          f"({100*total_mes/GRATIS_MES:.1f}% del cupo)")
    print(f"  restantes   : {max(0, GRATIS_MES-total_mes):>6,}")
    print(f"  COSTO       : USD {cobrable*USD_POR_MIL/1000:.2f}"
          + ("   <- dentro del cupo gratuito" if cobrable == 0 else "   <- SE PASO DEL CUPO"))

    barra = int(40 * min(1, total_mes / GRATIS_MES))
    print(f"\n  [{'#'*barra}{'.'*(40-barra)}]")

### Vigilar la cuota en la consola de Google Cloud

La cuenta de arriba es local. La oficial está en la consola, y hay tres cosas que ver y una que
conviene configurar.

**Ver el consumo** — [console.cloud.google.com/google/maps-apis/metrics](https://console.cloud.google.com/google/maps-apis/metrics)
Filtra por API (*Places API*) y por SKU. Muestra las llamadas por día y por método. Ojo: los datos
tardan unas horas en aparecer, así que no sirve para vigilar en tiempo real una corrida.

**Ver los límites** — [console.cloud.google.com/google/maps-apis/quotas](https://console.cloud.google.com/google/maps-apis/quotas)
Aquí aparece el uso contra los topes por día, por minuto y por usuario por minuto.

**Ver el gasto** — [console.cloud.google.com/billing](https://console.cloud.google.com/billing)
En *Reports*, filtrando por producto *Maps*, se ve cuánto se ha consumido del crédito.

**Poner un tope duro** ← esto es lo que de verdad te protege

En la página de *Quotas*, edita **«Requests per day»** de la Places API y ponlo en algo como
**4.500**. Es un límite que Google aplica: al alcanzarlo la API empieza a devolver error `429` en
vez de seguir cobrando. Nuestro bucle detecta ese código y se detiene solo.

Un presupuesto con alerta *avisa* después de gastar; un tope de cuota *impide* gastar. Con 3.517
llamadas necesarias y un tope de 4.500, es imposible pasarse por accidente.

**Alerta de presupuesto** (complementaria) — en *Billing → Budgets & alerts*, crea un presupuesto de
1 USD con aviso al 100 %. Si alguna vez llega un correo, algo se salió de lo previsto.

## Combinar los dos niveles

Para cada punto se arma un resumen con las siete categorías. Las del nivel 1 salen del catálogo por
distancia haversine; las del nivel 2, de la respuesta cacheada.

Este paso **no consume cuota**: trabaja sobre lo que ya está guardado, así que se puede repetir y
ajustar cuantas veces haga falta.

In [8]:
with sqlite3.connect(CACHE) as con:
    catalogo = pd.read_sql_query("SELECT * FROM catalogo", con)
    crudo = pd.read_sql_query(
        "SELECT punto, lat, lon, respuesta FROM puntos WHERE estado='ok'", con)

# Catalogo por categoria, como arrays para calcular distancias vectorizadas
cat_arrays = {c: (g["lat"].to_numpy(), g["lon"].to_numpy(), g["nombre"].to_numpy())
              for c, g in catalogo.groupby("categoria")}


def resumir(lat, lon, lugares):
    resumen = {}
    # --- nivel 1: distancia al catalogo, en local ---
    for cat, (clat, clon, cnom) in cat_arrays.items():
        d = metros(lat, lon, clat, clon)
        i = int(np.argmin(d))
        resumen[cat] = {"mas_cercano_m": round(float(d[i])),
                        "nombre": str(cnom[i]),
                        "n_1km": int((d <= 1000).sum())}
    # --- nivel 2: lo que devolvio la consulta del punto ---
    for lugar in lugares if isinstance(lugares, list) else []:
        loc = lugar.get("location") or {}
        if "latitude" not in loc:
            continue
        d = float(metros(lat, lon, loc["latitude"], loc["longitude"]))
        nombre = lugar.get("displayName", {}).get("text", "")
        for g in clasificar(lugar):
            actual = resumen.get(g)
            if actual is None:
                resumen[g] = {"mas_cercano_m": round(d), "nombre": nombre, "n_1km": int(d <= 1000)}
            else:
                actual["n_1km"] += int(d <= 1000)
                if d < actual["mas_cercano_m"]:
                    actual.update(mas_cercano_m=round(d), nombre=nombre)
    return resumen


resumenes = {r["punto"]: resumir(r["lat"], r["lon"], json.loads(r["respuesta"]))
             for _, r in crudo.iterrows()}
print(f"{len(resumenes):,} puntos resumidos\n")

if resumenes:
    categorias = list(cat_arrays) + list(GRUPOS_DENSOS)
    cob = pd.Series({c: 100 * np.mean([c in r for r in resumenes.values()])
                     for c in categorias}).round(1).sort_values(ascending=False)
    print("% de puntos con dato en cada categoria:")
    print(cob.to_string())
    print("\nLas del nivel 1 deben dar 100%: se calculan del catalogo, no dependen")
    print("de que el lugar sobreviva entre los 20 resultados de una busqueda.")

    clave, valor = next(iter(resumenes.items()))
    print(f"\nEjemplo — punto {clave}:")
    print(json.dumps(valor, indent=2, ensure_ascii=False))

25 puntos resumidos

% de puntos con dato en cada categoria:
hospital        100.0
mall            100.0
metro           100.0
universidad     100.0
colegio         100.0
parque          100.0
supermercado    100.0

Las del nivel 1 deben dar 100%: se calculan del catalogo, no dependen
de que el lugar sobreviva entre los 20 resultados de una busqueda.

Ejemplo — punto -33.435,-70.581:
{
  "hospital": {
    "mas_cercano_m": 2856,
    "nombre": "Colsalud - Medicina Funcional e Integrativa",
    "n_1km": 0
  },
  "mall": {
    "mas_cercano_m": 2883,
    "nombre": "RAACO",
    "n_1km": 0
  },
  "metro": {
    "mas_cercano_m": 552,
    "nombre": "Francisco Bilbao",
    "n_1km": 2
  },
  "universidad": {
    "mas_cercano_m": 2139,
    "nombre": "Ingenieros Comerciales UC",
    "n_1km": 0
  },
  "colegio": {
    "mas_cercano_m": 102,
    "nombre": "Casa Nativo",
    "n_1km": 12
  },
  "supermercado": {
    "mas_cercano_m": 126,
    "nombre": "OXXO",
    "n_1km": 4
  },
  "parque": {
    "mas_c

## Escribir en la base

Es el único paso que **modifica** `data/ofertas_unificado.sqlite`. Por eso:

1. Hace una **copia de respaldo** con fecha antes de tocar nada.
2. Agrega la columna con `ALTER TABLE ADD COLUMN`, que no altera los datos existentes.
3. Los avisos sin coordenadas quedan en `SIN_DATO`, la misma convención del resto de la base.

Está en `False` a propósito: revisa antes la cobertura de la celda anterior.

In [9]:
ESCRIBIR = False   # ponlo en True cuando la cobertura te convenza

if not ESCRIBIR:
    print("ESCRIBIR = False: la base no se toca.")
    listos = con_geo[con_geo["punto"].isin(resumenes)]
    print(f"Al activarlo se actualizarian {len(listos):,} avisos "
          f"({100*len(listos)/len(avisos):.1f}% del total).")
else:
    respaldo = DB.parent / f"{DB.stem}.bak-{datetime.now():%Y%m%d-%H%M}.sqlite"
    shutil.copy2(DB, respaldo)
    print(f"respaldo: {respaldo.name}")

    listos = con_geo[con_geo["punto"].isin(resumenes)].copy()
    listos["servicios"] = listos["punto"].map(
        lambda p: json.dumps(resumenes[p], ensure_ascii=False))

    with sqlite3.connect(DB, timeout=30) as con:
        columnas = {r[1] for r in con.execute("PRAGMA table_info(ofertas)")}
        if "servicios_cercanos" not in columnas:
            con.execute("ALTER TABLE ofertas ADD COLUMN servicios_cercanos TEXT "
                        "DEFAULT 'SIN_DATO'")
            print("columna servicios_cercanos creada")
        con.execute("UPDATE ofertas SET servicios_cercanos='SIN_DATO' "
                    "WHERE servicios_cercanos IS NULL")
        con.executemany(
            "UPDATE ofertas SET servicios_cercanos=? WHERE sitio=? AND id_aviso=?",
            listos[["servicios", "sitio", "id_aviso"]].itertuples(index=False, name=None))
        con.commit()
    print(f"{len(listos):,} avisos actualizados")

ESCRIBIR = False: la base no se toca.
Al activarlo se actualizarian 64 avisos (0.4% del total).


## Verificación

Comprobar que la columna quedó bien y que no se perdió ninguna fila.

In [10]:
with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as con:
    columnas = [r[1] for r in con.execute("PRAGMA table_info(ofertas)")]
    n = con.execute("SELECT COUNT(*) FROM ofertas").fetchone()[0]
    print(f"filas: {n:,}   columnas: {len(columnas)}")
    print(f"existe servicios_cercanos: {'servicios_cercanos' in columnas}")

    if "servicios_cercanos" in columnas:
        con_dato = con.execute("SELECT COUNT(*) FROM ofertas "
                               "WHERE servicios_cercanos != 'SIN_DATO'").fetchone()[0]
        print(f"avisos con servicios: {con_dato:,} ({100*con_dato/n:.1f}%)")
        fila = con.execute("SELECT comuna, servicios_cercanos FROM ofertas "
                           "WHERE servicios_cercanos != 'SIN_DATO' LIMIT 1").fetchone()
        if fila:
            print(f"\nEjemplo ({fila[0]}):")
            print(json.dumps(json.loads(fila[1]), indent=2, ensure_ascii=False))

filas: 15,808   columnas: 30
existe servicios_cercanos: False


---

## Siguiente paso

Con la columna poblada, el JSON se convierte en variables numéricas para los modelos:
`dist_metro_m`, `dist_colegio_m`, `n_colegios_1km`, `dist_parque_m`, `dist_supermercado_m`… Es ahí
donde el dato empieza a valer, porque son esas columnas las que entran al modelo.

Conviene entrenar **con y sin** esas variables y comparar el MdAPE. Es la única forma de saber
cuánto aportan realmente los servicios cercanos en vez de suponerlo — y el diagnóstico de
clustering ya dejó la línea base contra la que compararlas.